# Step 5.5: Improved BiLSTM+CRF Model Training
Enhanced version with attention and layer normalization

In [ ]:
import sys
run_cmd([sys.executable, "-m", "pip", "install", "-q", "tensorflow", "seqeval"])

In [ ]:
"""
Step 5.5: Improved BiLSTM+CRF Model Training
"""
import ast
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# ============ Load Data ============
print("Loading data...")
df = pd.read_csv('D:/Python/NLP/CK/final_data.csv')
df['tokens'] = df['tokens'].apply(ast.literal_eval)
df['labels'] = df['labels'].apply(ast.literal_eval)
df = df[df['tokens'].apply(len) == df['labels'].apply(len)].reset_index(drop=True)
print(f"Loaded {len(df)} sentences")

# Build label vocabulary
all_labels = set()
for labels in df['labels']:
    all_labels.update(labels)
label_list = sorted(list(all_labels))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
label_list = list(label_list)
print(f"Labels: {len(label_list)}")

# Build word vocabulary
all_words = []
for tokens in df['tokens']:
    all_words.extend(tokens)
word_counts = Counter(all_words)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in word_counts.most_common() if c >= 1]
word2id = {w: i for i, w in enumerate(vocab)}
print(f"Vocabulary: {len(vocab)} words")

# Save mappings
with open('D:/Python/NLP/CK/label_mappings.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f)

# ============ Prepare Sequences ============
MAX_LEN = 128

def encode(tokens, labels):
    token_ids = [word2id.get(w, word2id['<UNK>']) for w in tokens[:MAX_LEN]]
    label_ids = [label2id.get(l, 0) for l in labels[:MAX_LEN]]
    token_ids += [word2id['<PAD>']] * (MAX_LEN - len(token_ids))
    label_ids += [-100] * (MAX_LEN - len(label_ids))
    return token_ids, label_ids

X = []
y = []
for tokens, labels in zip(df['tokens'], df['labels']):
    tid, lid = encode(tokens, labels)
    X.append(tid)
    y.append(lid)

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)
print(f"Data shape: X={X.shape}, y={y.shape}")

indices = np.arange(len(X))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
# ============ Improved BiLSTM Model ============
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_improved_model(vocab_size, num_tags, 
                      word_embed_dim=300, lstm_units=256, dropout=0.3):
    """Improved BiLSTM with attention and layer normalization"""
    word_input = layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name='word_input')
    
    word_embed = layers.Embedding(vocab_size, word_embed_dim, mask_zero=True)(word_input)
    word_embed = layers.Dropout(dropout)(word_embed)
    
    lstm1 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))(word_embed)
    lstm1 = layers.LayerNormalization()(lstm1)
    lstm1 = layers.Dropout(dropout)(lstm1)
    
    lstm2 = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))(lstm1)
    lstm2 = layers.LayerNormalization()(lstm2)
    lstm2 = layers.Dropout(dropout)(lstm2)
    
    attn = layers.MultiHeadAttention(num_heads=4, key_dim=lstm_units)(lstm2, lstm2)
    attn = layers.Dropout(dropout)(attn)
    attn = layers.Add()([lstm2, attn])
    attn = layers.LayerNormalization()(attn)
    
    dense = layers.Dense(256, activation='relu')(attn)
    dense = layers.Dropout(dropout)(dense)
    logits = layers.Dense(num_tags, name='logits')(dense)
    
    model = keras.Model(inputs=word_input, outputs=logits)
    return model

print("\nBuilding improved BiLSTM model...")
model = build_improved_model(len(vocab), len(label_list))
model.summary()

In [ ]:
# ============ Loss Function ============
def masked_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_pred = tf.cast(y_pred, tf.float32)
    mask = tf.cast(y_true != -100, tf.float32)
    y_true_safe = tf.where(y_true == -100, tf.zeros_like(y_true), y_true)
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    loss = loss_fn(y_true_safe, y_pred)
    mask = tf.cast(mask, dtype=loss.dtype)
    loss = loss * mask
    return tf.reduce_sum(loss) / (tf.reduce_sum(mask) + 1e-8)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=masked_loss,
    metrics=['accuracy']
)

In [ ]:
# ============ Train ============
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    keras.callbacks.ModelCheckpoint('D:/Python/NLP/CK/bilstm_crf_improved.keras', monitor='val_loss', save_best_only=True)
]

print("\nTraining improved BiLSTM...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============ Evaluate ============
print("\n" + "="*50)
print("Evaluating on test set...")
logits = model.predict(X_test, batch_size=32)
predictions = np.argmax(logits, axis=-1)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
true_pred, true_label = [], []
for pred, label in zip(predictions, y_test):
    tp, tl = [], []
    for p, l in zip(pred, label):
        if l != -100:
            tp.append(id2label.get(int(p), 'O'))
            tl.append(id2label.get(int(l), 'O'))
    if tp:
        true_pred.append(tp)
        true_label.append(tl)

print("\n" + classification_report(true_label, true_pred, digits=4))
f1 = f1_score(true_label, true_pred)
print(f"\nTest F1: {f1:.4f}")

In [ ]:
# ============ Save & Test ============
model.save('D:/Python/NLP/CK/bilstm_crf_improved.keras')
print("\nModel saved!")

def predict(text):
    words = text.split()
    token_ids = [word2id.get(w, word2id['<UNK>']) for w in words]
    token_ids += [word2id['<PAD>']] * (MAX_LEN - len(token_ids))
    logits = model.predict(np.array([token_ids]), verbose=0)[0]
    pred_ids = np.argmax(logits, axis=-1)
    return [(w, id2label.get(int(p), 'O')) for w, p in zip(words, pred_ids[:len(words)])]

tests = ["Joe Biden met Donald Trump in Washington D.C. on Monday"]
for text in tests:
    print(f"\n{text}")
    for t, l in predict(text):
        print(f"  {t} -> {l}")